# Phase 6 — Abstractive Legal Summarisation with the pre-trained **GQLA** (decoder-only) model

GQLA has no encoder and no cross-attention, and none is added. In a decoder-only model, self-attention over a **prefix** does the job of cross-attention: every summary token attends to every earlier position. The problem is the 512-token window. Most judgments are far longer, so the document is first **compressed into a short memory**, and GQLA reads it as soft tokens:

```
document → 512-token chunks (K spread over the judgment)
        → frozen Phase-1 GQLA, run once      → hidden states of each chunk
        → m segment-means per chunk           → K·m memory slots  (cached fp16 on Drive)
        → MemoryBridge (trainable): chunk/slot positions + 1 bidirectional Transformer layer
                                   + projection into the token-embedding space
        → GQLA (adapted with the chosen method) reads, in ONE causal sequence of ≤ 512 positions:

          [ prompt? | pad … pad | mem_1 … mem_N | <s> | y_1 … y_L | </s> ]
                                                 └── loss only here (label smoothing 0.1)
```

**No re-pre-training is needed.** The only new weights are the bridge (~3M parameters) plus whatever the adaptation method adds. GQLA learns to read the memory during fine-tuning.

**Methods** (`method=`): `full` · `partial` · `lora` · `lorma` (LoRMA⁺, compulsory) · `qlora` (NF4 4-bit base + LoRA) · `houlsby` · `pfeiffer` · `prefix` · `prompt` · `frozen` (bridge only) · `kd` (5-layer student initialised from the fine-tuned teacher, CE + temperature-scaled KL on the teacher's logits). `mem_mode="lead"` is the plain prefix-LM baseline, where the first tokens of the document are fed as text.

**What changed relative to the Phase-5 GQLA notebook**
* Attention uses fused SDPA in training as well, with dropout passed to the kernel. The Phase-5 patch fell back to the full fp32 64-head score matrix whenever dropout was active, which is why full fine-tuning ran out of memory.
* The KV cache keeps the *un-rotated* RoPE key and rotates it at use time. The Phase-1 cache stored rotated keys and rotated them again on the next step.
* PEFT parameters are trainable and receive gradients. In Phase 5, the LoRA matrices were frozen by the wrapper and the forward pass ran under `no_grad`.

**Run order:** top to bottom. Cell 1b self-tests the new attention against the Phase-1 attention, cached against full decoding, and left padding. Cell 5 checks that every PEFT module is the identity at initialisation. Then run the smoke test, then the experiments. Memory features are extracted once per (split, K, m, strategy) and cached on Drive.

**Limits to keep in mind:** the summary has to fit in what the memory leaves of the 512 window (about 380 tokens with 128 memory slots). The config cell prints how many reference summaries are longer than that. If many are, reduce `mem_slots`/`mem_chunks`, or prefer the encoder–decoder notebook for long summaries.

In [ ]:
# metrics used for local validation (the official scorer computes BERTScore + ROUGE the same way)
!pip -q install rouge-score bert-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import os, json, math, time, random, copy, pickle, shutil, gc, glob
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
AMP_DTYPE = (None if device.type != "cuda" else
             torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)   # fp16 on T4
print("mixed precision:", AMP_DTYPE)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

# ---- paths (same layout as the Phase-5 GQLA notebook; edit if yours differ) -------------------------
cache_dir = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache"
ckpt_path = os.path.join(cache_dir, "gqla_checkpoints", "gqla_baseline_large.pt")      # Phase-1 best GQLA
tok_path  = os.path.join(cache_dir, "final_bpe_en.json")
phase1_nb = "/content/drive/MyDrive/Colab Notebooks/CS781_Capstone_Project_decoding.ipynb"
p6_dir    = os.path.join(cache_dir, "phase6_gqla"); os.makedirs(p6_dir, exist_ok=True)
local_ck  = "/content/p6_ckpt"; os.makedirs(local_ck, exist_ok=True)       # fast local disk; best ckpts copied to Drive

# ---- GQLA hyper-parameters: MUST equal what you pre-trained with ------------------------------------
GQLA_CFG = dict(d_model=512, n_layers=10, num_query_heads=64, num_kv_groups=32,
                head_dim=64, kv_latent_dim=96, rope_dim=32, path="gqa")
GQLA_CTX_LEN = 512                     # pre-training window: RoPE was never trained beyond this


def load_definitions_from_notebook(nb_path):
    """exec the definition cells of the Phase-1 notebook (tokenizer, normalisation, GQLA classes) so the
    architecture and tokenisation are byte-identical to pre-training."""
    nb = json.load(open(nb_path, encoding="utf-8"))
    wanted = ["class BPETokenizer", "def normalize_text", "class GQLARotaryEmbedding",
              "class GroupQueryLatentAttention", "class GQLAModel"]
    picked = []
    for m in wanted:
        for i, c in enumerate(nb["cells"]):
            if c["cell_type"] == "code" and m in "".join(c["source"]) and i not in picked:
                picked.append(i); break
        else:
            raise RuntimeError(f"could not find a cell containing {m!r} in {nb_path}")
    for i in sorted(picked):
        src = "".join(nb["cells"][i]["source"])
        src = "\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%")))
        exec(compile(src, f"<phase1 cell {i}>", "exec"), globals())
    print("loaded definitions from Phase-1 cells", sorted(picked))


load_definitions_from_notebook(phase1_nb)

if "GQLABlock" not in globals():           # normally defined next to GQLAModel; fallback = the Phase-5 copy
    print("GQLABlock not in the Phase-1 cells -> using the Phase-5 definition")

    class GQLABlock(nn.Module):
        def __init__(self, d_model, num_query_heads, num_kv_groups, head_dim, kv_latent_dim, rope_dim,
                     q_latent_dim=None, dropout=0.1, rope_base=10000.0, path="gqa", expansion_factor=4):
            super().__init__()
            self.attn = GroupQueryLatentAttention(d_model=d_model, num_query_heads=num_query_heads,
                                                  num_kv_groups=num_kv_groups, head_dim=head_dim,
                                                  kv_latent_dim=kv_latent_dim, rope_dim=rope_dim,
                                                  q_latent_dim=q_latent_dim, dropout=dropout,
                                                  rope_base=rope_base, path=path)
            self.feed_forward = nn.Sequential(nn.Linear(d_model, expansion_factor * d_model), nn.GELU(),
                                              nn.Linear(expansion_factor * d_model, d_model), nn.Dropout(dropout))
            self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
            self.dropout1, self.dropout2 = nn.Dropout(dropout), nn.Dropout(dropout)

        def forward(self, hidden_states, attention_mask=None, past_key_value=None, use_cache=False):
            a, new_past = self.attn(hidden_states=self.norm1(hidden_states), attention_mask=attention_mask,
                                    past_key_value=past_key_value, use_cache=use_cache)
            hidden_states = hidden_states + self.dropout1(a)
            hidden_states = hidden_states + self.dropout2(self.feed_forward(self.norm2(hidden_states)))
            return hidden_states, new_past

tokenizer = BPETokenizer.from_json(tok_path)
PAD_ID, BOS_ID, EOS_ID = tokenizer.stoi["<pad>"], tokenizer.stoi["<s>"], tokenizer.stoi["</s>"]
print(f"tokenizer vocab {len(tokenizer.itos):,} | PAD {PAD_ID} BOS {BOS_ID} EOS {EOS_ID}")


def load_pretrained_gqla():
    m = GQLAModel(vocab_size=len(tokenizer.itos), pad_token_id=PAD_ID, **GQLA_CFG)
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model_state_dict"])          # strict: fails loudly on a config mismatch
    return m.eval()


GQLA_BASE = load_pretrained_gqla()                     # pristine copy: experiments always deep-copy it
print(f"GQLA loaded: {sum(p.numel() for p in GQLA_BASE.parameters())/1e6:.1f}M params")

Mounted at /content/drive
device: cuda | Tesla T4
mixed precision: torch.float16
loaded definitions from Phase-1 cells [10, 22, 32, 33, 35]
tokenizer vocab 32,768 | PAD 0 BOS 2 EOS 3
GQLA loaded: 103.9M params


In [ ]:
# ============================================================
# DATA: Phase-6 (abstractive summarisation) from the course Hugging Face repo
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"


def find_repo_file(basename):
    """Locate e.g. train.json inside the Phase-6 folder without hard-coding its exact name."""
    files = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
    hits = [f for f in files if f.endswith("/" + basename) and ("summ" in f.lower() or "phase_6" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 6, got {hits}. "
                           f"Set the path by hand. Phase-6-like files: {[f for f in files if 'summ' in f.lower()]}")
    return hits[0]


_DOC_KEYS = ("document", "text", "judgment", "judgement", "doc", "source")
_SUM_KEYS = ("summary", "abstract", "target", "reference")


def _pick(r, keys, what, rid):
    for k in keys:
        if k in r and r[k] is not None:
            v = r[k]
            return " ".join(v) if isinstance(v, list) else str(v)
    if what == "summary":
        return None
    raise KeyError(f"record {rid}: no {what} field among {keys}; keys present: {list(r)}")


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):                          # {id: {...}} or {id: "text"}
        raw = [({**v, "id": k} if isinstance(v, dict) else {"id": k, "document": v}) for k, v in raw.items()]
    recs = []
    for r in raw:
        rid = str(r["id"])
        recs.append(dict(id=rid, doc=_pick(r, _DOC_KEYS, "document", rid), summ=_pick(r, _SUM_KEYS, "summary", rid)))
    return recs


# all_train_recs = read_split("train.json")
# test_recs = read_split("test.json")
# assert all(r["summ"] for r in all_train_recs), "some training records have no summary"
# print(f"{len(all_train_recs)} train documents | {len(test_recs)} test documents")

_DOC_KEYS = ("document", "text", "judgment", "judgement", "doc", "source", "input", "article", "case_text")
_SUM_KEYS = ("summary", "summaries", "abstract", "target", "reference", "headnote", "output", "gold", "label")


def _as_text(v):
    if v is None:
        return None
    if isinstance(v, list):
        v = " ".join(str(x) for x in v)
    v = str(v).strip()
    return v or None


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [({**v, "id": k} if isinstance(v, dict) else {"id": k, "document": v}) for k, v in raw.items()]
    keys = list(raw[0].keys())
    print(f"{filename}: {len(raw)} records | fields: {keys}")
    # document = first known doc key present, else the longest text field; summary = first known summary key present
    doc_k = next((k for k in _DOC_KEYS if k in keys), None)
    sum_k = next((k for k in _SUM_KEYS if k in keys), None)
    if doc_k is None:
        text_keys = [k for k in keys if k != "id" and isinstance(raw[0][k], (str, list))]
        doc_k = max(text_keys, key=lambda k: len(_as_text(raw[0][k]) or ""))
    print(f"  using document field '{doc_k}', summary field '{sum_k}'")
    return [dict(id=str(r.get("id", i)), doc=_as_text(r.get(doc_k)) or "",
                 summ=_as_text(r.get(sum_k)) if sum_k else None) for i, r in enumerate(raw)]


all_train_recs = read_split("train.json")
test_recs = read_split("test.json")
_n_missing = sum(r["summ"] is None for r in all_train_recs)
if _n_missing == len(all_train_recs):
    raise KeyError("no summary field found in train.json: look at the 'fields' printed above and add the "
                   "summary field's name to _SUM_KEYS")
if _n_missing:
    print(f"dropping {_n_missing} training records with an empty summary")
all_train_recs = [r for r in all_train_recs if r["summ"] is not None and r["doc"]]

# ---- no official dev split for Phase 6: hold out a validation split from train ----------------
VAL_FRAC, SPLIT_SEED = 0.05, 1234
_perm = np.random.RandomState(SPLIT_SEED).permutation(len(all_train_recs))
_n_val = max(50, int(round(VAL_FRAC * len(all_train_recs))))
val_recs = [all_train_recs[i] for i in _perm[:_n_val]]
train_recs = [all_train_recs[i] for i in _perm[_n_val:]]
print(f"split -> train {len(train_recs)} | val {len(val_recs)}  (val is held out from train.json)")

# ---- tokenise once, cache to Drive (no special tokens: added per chunk / per sequence later) ----
# NORMALIZE_TARGETS: apply the Phase-1 normalize_text to the reference summaries too. Keep True if
# normalize_text only cleans whitespace/unicode; set False if it lower-cases (the scorer compares
# against the raw references, so a lower-cased output would cost BERTScore).
NORMALIZE_TARGETS = True


def _tok_cached(texts, tag, norm=True, bs=1000, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p6_dir, f"tok_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(texts), bs):
        part = [normalize_text(t) if norm else t for t in texts[i:i + bs]]
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(part, add_special_tokens=False, num_workers=workers, chunk_size=16)]
        print(f"  tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


def _tok_split(recs, tag):
    docs = _tok_cached([r["doc"] for r in recs], f"{tag}_doc_s{SPLIT_SEED}")
    sums = None
    if recs[0]["summ"] is not None:
        sums = _tok_cached([r["summ"] for r in recs], f"{tag}_sum_s{SPLIT_SEED}_n{int(NORMALIZE_TARGETS)}",
                           norm=NORMALIZE_TARGETS)
    return docs, sums


tr_doc, tr_sum = _tok_split(train_recs, "train")
va_doc, va_sum = _tok_split(val_recs, "val")
te_doc, _ = _tok_split(test_recs, "test")

# ---- length statistics: these decide the generation lengths and the context layout ------------
dl = np.array([len(x) for x in tr_doc]); sl = np.array([len(x) for x in tr_sum])
sw = np.array([len(r["summ"].split()) for r in train_recs])
pct = lambda a: "  ".join(f"p{p} {int(np.percentile(a, p))}" for p in (10, 50, 90, 95, 99))
print("document BPE tokens :", pct(dl), f" max {dl.max()}")
print("summary  BPE tokens :", pct(sl), f" max {sl.max()}")
print("summary  words      :", pct(sw))
print(f"tokens per summary word ~ {sl.sum() / max(sw.sum(), 1):.2f} | summary/document length ratio median "
      f"{np.median(sl / np.maximum(dl, 1)):.3f}")
print(f"512-token chunks per document: median {np.median(np.ceil(dl / 511)):.0f}  p90 {np.percentile(np.ceil(dl / 511), 90):.0f}")

SUM_TOK_P10, SUM_TOK_P50, SUM_TOK_P90, SUM_TOK_P95 = (int(np.percentile(sl, p)) for p in (10, 50, 90, 95))
SUM_WORDS_P50 = int(np.median(sw))

train.json: 7030 records | fields: ['id', 'document', 'summary']
  using document field 'document', summary field 'summary'
test.json: 100 records | fields: ['id', 'document']
  using document field 'document', summary field 'None'
dropping 2 training records with an empty summary
split -> train 6677 | val 351  (val is held out from train.json)
  tokenised 1,000/6,677 (train_doc_s1234)
  tokenised 2,000/6,677 (train_doc_s1234)
  tokenised 3,000/6,677 (train_doc_s1234)
  tokenised 4,000/6,677 (train_doc_s1234)
  tokenised 5,000/6,677 (train_doc_s1234)
  tokenised 6,000/6,677 (train_doc_s1234)
  tokenised 6,677/6,677 (train_doc_s1234)
  tokenised 1,000/6,677 (train_sum_s1234_n1)
  tokenised 2,000/6,677 (train_sum_s1234_n1)
  tokenised 3,000/6,677 (train_sum_s1234_n1)
  tokenised 4,000/6,677 (train_sum_s1234_n1)
  tokenised 5,000/6,677 (train_sum_s1234_n1)
  tokenised 6,000/6,677 (train_sum_s1234_n1)
  tokenised 6,677/6,677 (train_sum_s1234_n1)
  tokenised 351/351 (val_doc_s1234)
  tokeni

In [ ]:
# ============================================================
# Model-agnostic utilities: chunking, detokenisation, post-processing, ROUGE / BERTScore, submission
# ============================================================
import re, zipfile, collections


# ---------------- long-document chunking (as in Phase 5) ----------------
def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="uniform"):
    """uniform: spread the budget over the whole judgment (facts at the start, reasoning in the middle,
    the order at the end all matter for a summary). head / tail / head_tail kept for the ablation."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(strategy)


# ---------------- detokenisation + post-processing ----------------
_FIXES = [(re.compile(r"\s+([.,;:!?%)\]}'])"), r"\1"),      # 'court .'  -> 'court.'
          (re.compile(r"([(\[{])\s+"), r"\1"),               # '( a )'    -> '(a)'
          (re.compile(r"(?<=\w)\s*([-/])\s*(?=\w)"), r"\1"),  # '302 / 34' -> '302/34', 'Section - 3' -> 'Section-3'
          (re.compile(r"\s{2,}"), " ")]


def detok(ids):
    """BPE ids -> text. tokenizer.decode() puts a space after every word-final token including punctuation
    ('court . the'), so spacing is repaired here. Special tokens are dropped."""
    ids = [int(i) for i in ids if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
    s = tokenizer.decode(ids)
    for pat, rep in _FIXES:
        s = pat.sub(rep, s)
    return s.strip()


def postprocess(text, max_words=None):
    """drop exact/near-exact repeated sentences (small models loop), cap length, end on a sentence boundary"""
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    seen, out = set(), []
    for s in sents:
        key = re.sub(r"\W+", " ", s.lower()).strip()
        if len(key) < 2 or key in seen:
            continue
        seen.add(key); out.append(s)
    text = " ".join(out)
    if max_words:
        w = text.split()
        if len(w) > max_words:
            text = " ".join(w[:max_words])
    cut = max(text.rfind(". "), text.rfind("."))
    if cut > 0.8 * len(text) and cut < len(text) - 1:     # trailing fragment after the last full stop
        text = text[:cut + 1]
    return text.strip() or "."


# ---------------- metrics ----------------
try:
    from rouge_score import rouge_scorer as _rs
    _ROUGE = _rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
except Exception:
    _ROUGE = None


def _toks(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def _f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def _rouge_n(a, b, n):
    ca = collections.Counter(zip(*[a[i:] for i in range(n)])); cb = collections.Counter(zip(*[b[i:] for i in range(n)]))
    ov = sum((ca & cb).values())
    return _f1(ov / max(sum(ca.values()), 1), ov / max(sum(cb.values()), 1))


def _rouge_l(a, b):
    if not a or not b:
        return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]
    return _f1(l / len(a), l / len(b))


def rouge_scores(preds, refs):
    r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        if _ROUGE is not None:
            s = _ROUGE.score(r, p)
            r1 += s["rouge1"].fmeasure; r2 += s["rouge2"].fmeasure; rl += s["rougeL"].fmeasure
        else:                                            # fallback (no stemming): numbers differ slightly
            a, b = _toks(p), _toks(r)
            r1 += _rouge_n(a, b, 1); r2 += _rouge_n(a, b, 2); rl += _rouge_l(a, b)
    n = max(len(preds), 1)
    return dict(rouge1=r1 / n, rouge2=r2 / n, rougeL=rl / n)


BERTSCORE_KW = dict(lang="en")      # the course scorer's settings are not published; lang="en" = roberta-large
_BS = {}


def bertscore_f1(preds, refs, batch_size=16):
    """Evaluation only (the official scorer does the same). Returns None if bert-score is unavailable."""
    try:
        from bert_score import BERTScorer
    except Exception:
        return None
    key = json.dumps(BERTSCORE_KW, sort_keys=True)
    if key not in _BS:
        _BS[key] = BERTScorer(device=str(device), **BERTSCORE_KW)
    _, _, f = _BS[key].score(preds, refs, batch_size=batch_size)
    return float(f.mean())


def evaluate_texts(preds, refs, bert=True):
    m = rouge_scores(preds, refs)
    m["bertscore"] = bertscore_f1(preds, refs) if bert else None
    m["avg_words"] = float(np.mean([len(p.split()) for p in preds]))
    m["ref_words"] = float(np.mean([len(r.split()) for r in refs]))
    return m


def fmt_metrics(m):
    bs = "n/a" if m.get("bertscore") is None else f"{m['bertscore']:.4f}"
    return (f"BERTScore {bs} | R1 {m['rouge1']:.4f} R2 {m['rouge2']:.4f} RL {m['rougeL']:.4f} | "
            f"words {m['avg_words']:.0f} (ref {m['ref_words']:.0f})")


# ---------------- submission ----------------
def write_submission(preds_by_id, name):
    out = [dict(id=r["id"], summary=preds_by_id[r["id"]]) for r in test_recs]
    assert len(out) == len(test_recs) == len({o["id"] for o in out}), "need exactly one summary per test id"
    assert all(isinstance(o["summary"], str) and o["summary"].strip() for o in out), "empty summary"
    d = os.path.join(p6_dir, f"submission_{name}"); os.makedirs(d, exist_ok=True)
    jp = os.path.join(d, "prediction.json")
    json.dump(out, open(jp, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(d, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")          # at the ROOT of the zip, as Codabench requires
    print(f"wrote {len(out)} summaries (avg {np.mean([len(o['summary'].split()) for o in out]):.0f} words) -> {zp}")
    return zp

In [ ]:
# ============================================================
# Generic batched beam search (model-agnostic)
#
# The model side supplies three closures:
#   start()            -> (state, logits[B,V])   consume the prompt / encode the source, logits for the 1st token
#   step(state, tok)   -> (state, logits[N,V])   consume one token per hypothesis (N = B*beams)
#   reorder(state, ix) -> state                  keep hypothesis rows ix (rows never move to another source:
#                                                row r always belongs to source r // beams)
# Features: length penalty, min length, no-repeat-ngram blocking, repetition penalty, early stopping.
# num_beams=1 gives greedy decoding.
# ============================================================
@torch.no_grad()
def beam_search(start, step, reorder, num_beams=4, max_new=256, min_new=0, len_penalty=1.0,
                no_repeat_ngram=3, rep_penalty=1.0, eos_id=None, banned=()):
    eos_id = EOS_ID if eos_id is None else eos_id
    state, logits = start()
    B, V = logits.shape
    K, dev = num_beams, logits.device
    state = reorder(state, torch.arange(B, device=dev).repeat_interleave(K))
    logits = logits.repeat_interleave(K, 0)
    beam_scores = torch.zeros(B, K, device=dev)
    beam_scores[:, 1:] = -1e9                       # identical beams at t=0: keep only the first
    seqs = [[] for _ in range(B * K)]
    finished = [[] for _ in range(B)]               # (normalised score, tokens)
    done = [False] * B
    banned = [t for t in banned if t is not None and t != eos_id]

    for t in range(max_new):
        lp = torch.log_softmax(logits.float(), -1)
        if banned:
            lp[:, banned] = -float("inf")
        if t < min_new:
            lp[:, eos_id] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            for r, s in enumerate(seqs):
                if s:
                    ix = torch.tensor(sorted(set(s)), device=dev)
                    lp[r, ix] -= math.log(rep_penalty)
        n = no_repeat_ngram
        if n and t >= n - 1:
            for r, s in enumerate(seqs):
                if done[r // K]:
                    continue
                prefix = tuple(s[len(s) - n + 1:]) if n > 1 else ()
                bad = [s[i + n - 1] for i in range(len(s) - n + 1) if tuple(s[i:i + n - 1]) == prefix]
                if bad:
                    lp[r, bad] = -float("inf")

        cand = (beam_scores.view(-1, 1) + lp).view(B, K * V)
        top_s, top_i = cand.topk(2 * K, dim=1)
        top_s, top_i = top_s.tolist(), top_i.tolist()
        sel_rows, sel_tok, new_scores = [], [], []
        for b in range(B):
            chosen = []
            if not done[b]:
                for sc, ix in zip(top_s[b], top_i[b]):
                    beam, tok = divmod(ix, V)
                    if sc < -1e8:
                        break
                    if tok == eos_id:
                        finished[b].append((sc / ((t + 1) ** len_penalty), list(seqs[b * K + beam])))
                    else:
                        chosen.append((b * K + beam, tok, sc))
                    if len(chosen) == K:
                        break
                # early stop: K finished hypotheses and no live beam can beat the worst of them
                if len(finished[b]) >= K:
                    best_live = max((c[2] for c in chosen), default=-1e9) / ((t + 1) ** len_penalty)
                    if best_live <= sorted(f[0] for f in finished[b])[-K]:
                        done[b] = True
            while len(chosen) < K:                    # finished source or not enough candidates: dummy rows
                chosen.append((b * K, eos_id if done[b] else (chosen[0][1] if chosen else eos_id), -1e9))
            for row, tok, sc in chosen:
                sel_rows.append(row); sel_tok.append(tok); new_scores.append(sc)
        if all(done):
            break
        seqs = [seqs[r] + [tk] for r, tk in zip(sel_rows, sel_tok)]
        beam_scores = torch.tensor(new_scores, device=dev).view(B, K)
        rows = torch.tensor(sel_rows, device=dev)
        state = reorder(state, rows)
        state, logits = step(state, torch.tensor(sel_tok, device=dev))

    out = []
    L = len(seqs[0]) if seqs else 0
    for b in range(B):
        if not done[b]:
            for k in range(K):
                sc = float(beam_scores[b, k])
                if sc > -1e8:
                    finished[b].append((sc / (max(L, 1) ** len_penalty), seqs[b * K + k]))
        out.append(max(finished[b], key=lambda x: x[0])[1] if finished[b] else [])
    return out

In [ ]:
# ============================================================
# Parameter-efficient adaptation modules (own implementations, no peft library)
#   LoRA     (Hu et al., 2021)            W x + (a/r) B A x                      B = 0 at init
#   LoRMA+   (Bihany, Patel, Modi, 2025)   (I + (a/r) B A) W0 x                   multiplicative, additive rank
#                                          inflation; B = 0 at init -> identity; computed as y0 + s*B(A y0),
#                                          y0 = W0 x, so no d x d matrix is ever formed (the paper's re-ordering)
#   QLoRA    (Dettmers et al., 2023)       frozen base stored as 4-bit NF4 (blockwise absmax), LoRA on top
#   Adapters (Houlsby et al., 2019 / Pfeiffer et al., 2021)  bottleneck MLP, zero-init up-projection
#   Prefix   (Li & Liang, 2021)            learned per-layer key/value prefixes (built in the model code)
#   Prompt   (Lester et al., 2021)         learned soft input tokens (built in the model code)
# ============================================================
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))


class LoRMALinear(nn.Module):
    """LoRMA+ : W = (I_out + (a/r) B A) W0 ;  B in R^{out x r}, A in R^{r x out}. The bias is added after the
    multiplicative transform (it is not part of W0)."""

    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A.to(y0.dtype)), self.B.to(y0.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(y.dtype)


NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                         -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                         0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                         0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer stored as 4-bit NormalFloat codes (2 per byte) + one fp16 absmax per block of 64.
    Dequantised on the fly in forward (memory saving of QLoRA, without bitsandbytes)."""

    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float()
        self.shape, self.block = tuple(W.shape), block
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-8)
        code = NF4_CODE.to(flat.device)
        idx = torch.empty(flat.shape, dtype=torch.uint8, device=flat.device)
        for i in range(0, flat.size(0), 4096):                       # chunked argmin keeps memory small
            idx[i:i + 4096] = ((flat[i:i + 4096] / absmax[i:i + 4096]).unsqueeze(-1) - code).abs().argmin(-1)
        idx = idx.view(-1)
        if idx.numel() % 2:
            idx = torch.cat([idx, idx.new_zeros(1)])
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("code", code.clone())
        self.n = W.numel()
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.in_features, self.out_features = self.shape[1], self.shape[0]

    def dequant(self, dtype):
        idx = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = self.code[idx][: self.absmax.numel() * self.block].view(-1, self.block) * self.absmax.float()
        return w.view(-1)[: self.n].view(self.shape).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class Adapter(nn.Module):
    """bottleneck adapter: x + up(act(down(x))); up is zero-initialised so the model starts unchanged"""

    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.drop(self.up(F.gelu(self.down(x))))


class AdapterAfter(nn.Module):
    """Applies an adapter to the output of a sub-layer (Houlsby: after attention and FFN; Pfeiffer: after FFN).
    Works for sub-layers that return a tensor or a tuple (tensor, cache)."""

    def __init__(self, inner, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck, dropout)

    def forward(self, *a, **k):
        out = self.inner(*a, **k)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def replace_linears(root, make, keep):
    """Replace every nn.Linear child c (full name n) under root with make(c) when keep(n) is True."""
    done = []
    for name, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            full = f"{name}.{cname}" if name else cname
            if type(child) is nn.Linear and keep(full):
                setattr(mod, cname, make(child)); done.append(full)
    return done


def n_params(m, trainable=True):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False

In [ ]:
# ============================================================
# Training / evaluation loop (shared by every method; the model file supplies the Summarizer)
#
# A Summarizer exposes:
#   sm.loss(batch, teacher=None)      -> scalar loss (teacher != None -> knowledge distillation)
#   sm.generate(batch, gen)           -> list of generated token-id lists
#   sm.pretrained_param_ids           -> ids of parameters that came from the Phase-1 checkpoint
# and the notebook supplies make_loader(split, cfg, shuffle, gen=False).
# ============================================================
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def amp_ctx(on=True):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=bool(on and AMP_DTYPE is not None))


def trainable_state(sm):
    return {n: p.detach().cpu().clone() for n, p in sm.named_parameters() if p.requires_grad}


def save_trainable(sm, path, extra=None):
    torch.save({"trainable": trainable_state(sm), **(extra or {})}, path)


def load_trainable(sm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    missing = [n for n, p in sm.named_parameters() if p.requires_grad and n not in ck["trainable"]]
    if missing:
        raise RuntimeError(f"checkpoint {path} lacks {len(missing)} trainable tensors, e.g. {missing[:3]} "
                           f"(was it saved with the same method/config?)")
    sm.load_state_dict(ck["trainable"], strict=False)
    return ck


@torch.no_grad()
def eval_loss(sm, loader, max_batches=None):
    sm.eval(); tot, n = 0.0, 0
    for i, b in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        with amp_ctx():
            tot += float(sm.loss(b)); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def generate_split(sm, split, cfg, n_docs=None, gen=None):
    """-> {id: summary text}"""
    sm.eval()
    gen = {**GEN, **(gen or {})}
    loader = make_loader(split, cfg, shuffle=False, gen=True, limit=n_docs)
    out, t0 = {}, time.time()
    for b in loader:
        with amp_ctx():
            seqs = sm.generate(b, gen)
        for rid, s in zip(b["ids"], seqs):
            out[rid] = postprocess(detok(s), gen.get("max_words"))
        if len(out) % (len(b["ids"]) * 10) == 0:
            print(f"    generated {len(out)} ({time.time() - t0:.0f}s)")
    return out


def score_split(preds, recs, bert=True):
    refs = {r["id"]: r["summ"] for r in recs}
    ids = [i for i in preds if i in refs]
    return evaluate_texts([preds[i] for i in ids], [refs[i] for i in ids], bert=bert)


def train_summarizer(sm, cfg, tag, teacher=None):
    tr, va = make_loader("train", cfg, shuffle=True), make_loader("val", cfg, shuffle=False)
    pre = sm.pretrained_param_ids
    new_p = [p for p in sm.parameters() if p.requires_grad and id(p) not in pre]
    old_p = [p for p in sm.parameters() if p.requires_grad and id(p) in pre]
    groups = ([dict(params=new_p, lr=cfg.lr)] if new_p else []) + ([dict(params=old_p, lr=cfg.lr_backbone)] if old_p else [])
    if not groups:
        raise RuntimeError("nothing to train")
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd, betas=(0.9, 0.98))
    total = max(1, cfg.epochs * math.ceil(len(tr) / cfg.accum))
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))))
    use_scaler = AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    best, bad, step = -1.0, 0, 0
    ck_path = os.path.join(local_ck, f"{tag}.pt")
    if teacher is not None:
        teacher.eval(); freeze(teacher)
    print(f"[{tag}] {len(tr)} batches/epoch x {cfg.epochs} epochs | accum {cfg.accum} | "
          f"trainable new {sum(p.numel() for p in new_p)/1e6:.2f}M + backbone {sum(p.numel() for p in old_p)/1e6:.2f}M")
    for ep in range(cfg.epochs):
        sm.train(); opt.zero_grad(set_to_none=True)
        t0, run = time.time(), 0.0
        for i, b in enumerate(tr):
            with amp_ctx():
                loss = sm.loss(b, teacher=teacher)
            if not torch.isfinite(loss):
                print(f"[{tag}] non-finite loss at batch {i}, skipped"); opt.zero_grad(set_to_none=True); continue
            scaler.scale(loss / cfg.accum).backward()
            run += float(loss)
            if (i + 1) % cfg.accum == 0 or i + 1 == len(tr):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if (i + 1) % cfg.log_every == 0:
                print(f"[{tag}] ep {ep+1} batch {i+1}/{len(tr)} loss {run/(i+1):.4f} ({time.time()-t0:.0f}s)")
        vl = eval_loss(sm, va, max_batches=cfg.val_loss_batches)
        preds = generate_split(sm, "val", cfg, n_docs=cfg.n_eval_gen)
        m = score_split(preds, val_recs, bert=cfg.select_metric == "bertscore")
        score = m[cfg.select_metric] if m.get(cfg.select_metric) is not None else m["rougeL"]
        print(f"[{tag}] ep {ep+1:02d} train {run/len(tr):.4f} | val loss {vl:.4f} | {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
        if score > best:
            best, bad = score, 0
            save_trainable(sm, ck_path, {"cfg": vars(cfg), "epoch": ep + 1, "val": m})
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    load_trainable(sm, ck_path)
    shutil.copy(ck_path, os.path.join(p6_dir, f"ckpt_{tag}.pt"))          # persist the best checkpoint to Drive
    return best

In [ ]:
# ============================================================
# 1. GQLA ATTENTION FOR SUMMARISATION (replaces GroupQueryLatentAttention.forward; same weights, same maths)
#
# Why a new forward:
#   * fused SDPA attention in train AND eval (dropout passed to the kernel). The Phase-5 patch fell back to the
#     O(T^2 * 64 heads) fp32 manual attention whenever dropout was active, which is what OOM'd full fine-tuning.
#   * KV cache stores the GROUPED k_C / v_C (32 groups) and the UN-rotated RoPE key; RoPE is applied with the
#     absolute position at use time. (The Phase-1 cache stored the rotated key and rotated it again on the next
#     step, so cached generation drifted.)
#   * key-padding mask (left-padded memory), with every query always allowed to see itself so padded rows
#     never become all -inf (NaN through the values).
#   * optional learned prefix (prefix tuning): P hidden vectors per layer pushed through the layer's own frozen
#     W_DKV / W_UK / W_UV / W_KR, so the extra keys/values live in the same space as real ones.
# ============================================================
GQLA_CTX = {"key_pad": None}            # [B, S_total] bool (True = real key), set by gqla_hidden()

if not hasattr(GroupQueryLatentAttention, "_orig_forward"):
    GroupQueryLatentAttention._orig_forward = GroupQueryLatentAttention.forward


def _rope(x, pos, inv_freq):
    """x [B,h,T,d], pos [T] absolute positions; rotation computed in fp32 (fp16 angles are too coarse)."""
    with torch.autocast(device_type=x.device.type, enabled=False):
        f = pos.float()[:, None] * inv_freq.float()[None, :]
        emb = torch.cat([f, f], -1)
        cos, sin = emb.cos()[None, None], emb.sin()[None, None]
        xf = x.float()
        half = xf.shape[-1] // 2
        rot = torch.cat([-xf[..., half:], xf[..., :half]], -1)
        out = xf * cos + rot * sin
    return out.to(x.dtype)


def gqla_attn_forward(self, hidden_states, attention_mask=None, past_key_value=None, use_cache=False):
    assert attention_mask is None, "pass padding through GQLA_CTX['key_pad'], not attention_mask"
    assert self.path == "gqa"
    x = hidden_states
    B, T, _ = x.shape
    H, G, D, R = self.num_query_heads, self.num_kv_groups, self.head_dim, self.rope_dim
    hp = self.heads_per_group
    c_Q, c_KV = self.W_DQ(x), self.W_DKV(x)
    q_C = self.W_UQ(c_Q).view(B, T, H, D).transpose(1, 2)
    k_C = self.W_UK(c_KV).view(B, T, G, D).transpose(1, 2)
    v_C = self.W_UV(c_KV).view(B, T, G, D).transpose(1, 2)
    q_R = self.W_QR(c_Q).view(B, T, 1, R).transpose(1, 2)
    k_R = self.W_KR(x).view(B, T, 1, R).transpose(1, 2)                      # un-rotated
    past = 0
    if past_key_value is not None:
        pk, pv, pr = past_key_value
        past = pk.size(2)
        k_C, v_C, k_R = torch.cat([pk, k_C], 2), torch.cat([pv, v_C], 2), torch.cat([pr, k_R], 2)
    S = k_C.size(2)
    inv = self.rotary_emb.inv_freq
    qR = _rope(q_R, torch.arange(past, past + T, device=x.device), inv)
    kR = _rope(k_R, torch.arange(S, device=x.device), inv)
    q = torch.cat([q_C, qR.expand(-1, H, -1, -1)], -1)
    k = torch.cat([k_C.repeat_interleave(hp, 1), kR.expand(-1, H, -1, -1)], -1)
    v = v_C.repeat_interleave(hp, 1)

    P = 0
    pre = getattr(self, "prefix", None)                                       # nn.Parameter [P, d_model] or None
    if pre is not None:
        P = pre.size(0)
        xp = pre.to(x.dtype)
        cp = self.W_DKV(xp)
        pk_ = torch.cat([self.W_UK(cp).view(P, G, D).transpose(0, 1).repeat_interleave(hp, 0),
                         self.W_KR(xp).view(1, P, R).expand(H, -1, -1)], -1)       # [H,P,D+R], no position
        pv_ = self.W_UV(cp).view(P, G, D).transpose(0, 1).repeat_interleave(hp, 0)
        k = torch.cat([pk_.unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype), k], 2)
        v = torch.cat([pv_.unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype), v], 2)

    kp = GQLA_CTX["key_pad"]                                                  # [B,S] or None
    if kp is not None:
        assert kp.shape == (B, S), f"key_pad {tuple(kp.shape)} != {(B, S)}"
    is_causal, mask = False, None
    if kp is None and P == 0 and past == 0 and T > 1:
        is_causal = True
    elif T > 1 or kp is not None or P > 0:
        qpos = torch.arange(past, past + T, device=x.device)[:, None]
        kpos = torch.arange(S, device=x.device)[None, :]
        m = (kpos <= qpos)[None, None]                                        # causal  [1,1,T,S]
        if kp is not None:
            m = (m & kp[:, None, None, :]) | (kpos == qpos)[None, None]       # padding; self always visible
        if P:
            m = torch.cat([m.new_ones(m.shape[:-1] + (P,)).expand(m.size(0), -1, -1, -1), m], -1)
        mask = m
    drop = self.attn_dropout.p if self.training else 0.0
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop, is_causal=is_causal)
    out = self.W_O(o.transpose(1, 2).reshape(B, T, H * D))
    return out, ((k_C, v_C, k_R) if use_cache else None)


GroupQueryLatentAttention.forward = gqla_attn_forward


def gqla_hidden(model, ids=None, embeds=None, key_pad=None, past=None, use_cache=False, ckpt=False, upto=None):
    """Run the GQLA stack -> final hidden states (after norm_final) [B,T,d] and the new cache.
    embeds: soft inputs (memory / prompt) instead of ids.  key_pad: [B, past+T] bool.
    upto: stop after block `upto` (intermediate-layer features, parameter-free LayerNorm).
    Bottom blocks with no trainable parameters run under no_grad when the input needs no gradient."""
    h = model.token_embedding(ids) if embeds is None else embeds
    layers = model.layers if upto is None else model.layers[:upto + 1]
    lo = 0
    if torch.is_grad_enabled() and not h.requires_grad:
        while lo < len(layers) and not any(p.requires_grad for p in layers[lo].parameters()):
            lo += 1
    new = []
    for i, blk in enumerate(layers):
        p = None if past is None else past[i]

        def run(x, blk=blk, p=p, kp=key_pad):
            GQLA_CTX["key_pad"] = kp
            return blk(hidden_states=x, past_key_value=p, use_cache=use_cache)

        if i < lo:
            with torch.no_grad():
                h, c = run(h)
        elif ckpt and torch.is_grad_enabled() and not use_cache:
            h = grad_ckpt(lambda x, run=run: run(x)[0], h, use_reentrant=False); c = None
        else:
            h, c = run(h)
        new.append(c)
    GQLA_CTX["key_pad"] = None
    h = model.norm_final(h) if upto is None else F.layer_norm(h, (h.size(-1),))
    return h, (new if use_cache else None)


def reorder_cache(past, idx):
    return [tuple(t.index_select(0, idx) for t in layer) for layer in past]

In [ ]:
# ============================================================
# 1b. SELF-TEST (seconds). Run before any training: proves the new attention reproduces the Phase-1 model,
#     that cached incremental decoding == full forward, and that left padding is harmless.
# ============================================================
@torch.no_grad()
def check_gqla_patch():
    m = GQLA_BASE.to(device).eval()
    docs = [d for d in tr_doc if len(d) >= 300][:4]
    ids = torch.as_tensor(np.stack([d[:256] for d in docs]), dtype=torch.long, device=device)

    # (1) new attention == original Phase-1 attention (fp32)
    GroupQueryLatentAttention.forward = GroupQueryLatentAttention._orig_forward
    ref = m.norm_final(_run_blocks_original(m, ids))
    GroupQueryLatentAttention.forward = gqla_attn_forward
    new, _ = gqla_hidden(m, ids)
    d1 = (ref - new).abs().max().item()
    print(f"(1) patched vs original attention      max|diff| {d1:.2e}")
    assert d1 < 1e-3 * max(1.0, ref.abs().max().item()), "patched attention does not match the original"

    # (2) left-padded prompt + cached incremental decoding == one full forward
    B, Tp, Tg = ids.size(0), 64, 24
    pad = [0, 7, 13, 30][:B]                                                 # different left-pad per row
    full = torch.full((B, Tp + Tg), PAD_ID, dtype=torch.long, device=device)
    kp = torch.zeros(B, Tp + Tg, dtype=torch.bool, device=device)
    for b in range(B):
        full[b, pad[b]:] = ids[b, : Tp + Tg - pad[b]]; kp[b, pad[b]:] = True
    h_full, _ = gqla_hidden(m, full, key_pad=kp)
    h_p, past = gqla_hidden(m, full[:, :Tp], key_pad=kp[:, :Tp], use_cache=True)
    steps = [h_p[:, -1]]
    for t in range(Tp, Tp + Tg - 1):
        h_t, past = gqla_hidden(m, full[:, t:t + 1], key_pad=kp[:, :t + 1], past=past, use_cache=True)
        steps.append(h_t[:, -1])
    inc = torch.stack(steps, 1)
    d2 = (inc - h_full[:, Tp - 1:Tp + Tg - 1]).abs().max().item()
    print(f"(2) cached decoding vs full forward     max|diff| {d2:.2e}")
    assert d2 < 1e-3 * max(1.0, h_full.abs().max().item()), "KV-cache decoding diverges from the full forward"

    # (3) left padding does not change the real positions (row with pad 30 vs the same text unpadded)
    solo, _ = gqla_hidden(m, full[B - 1:, pad[-1]:])
    d3 = (solo[0] - h_full[B - 1, pad[-1]:]).abs().max().item()
    print(f"(3) left-padded row vs unpadded row     max|diff| {d3:.2e}  | NaN: {bool(torch.isnan(h_full).any())}")
    assert d3 < 1e-3 * max(1.0, solo.abs().max().item()) and not torch.isnan(h_full).any()

    # (4) mixed precision closeness
    with amp_ctx():
        h16, _ = gqla_hidden(m, ids)
    cos = F.cosine_similarity(h16.float().reshape(-1, h16.size(-1)), new.reshape(-1, new.size(-1)), -1).min().item()
    print(f"(4) autocast vs fp32                   min cosine {cos:.5f}")
    print("GQLA self-test passed")


def _run_blocks_original(m, ids):
    h = m.token_embedding(ids)
    for blk in m.layers:
        h, _ = blk(hidden_states=h)
    return h


@torch.no_grad()
def check_peft_identity():
    """LoRA / LoRMA / adapters must leave the model unchanged at initialisation (B=0 / zero up-projection)."""
    ids = torch.as_tensor(np.stack([d[:128] for d in tr_doc if len(d) >= 128][:2]), dtype=torch.long, device=device)
    ref, _ = gqla_hidden(GQLA_BASE.to(device).eval(), ids)
    for meth in ("lora", "lorma", "houlsby", "pfeiffer"):
        g = copy.deepcopy(GQLA_BASE)
        apply_backbone_method(g, SimpleNamespace(**{**BASE_CFG, "method": meth}))
        g = g.to(device).eval()
        out, _ = gqla_hidden(g, ids)
        print(f"  {meth:9s} identity at init: max|diff| {(out - ref).abs().max().item():.2e}")
        del g
    g = copy.deepcopy(GQLA_BASE)
    apply_backbone_method(g, SimpleNamespace(**{**BASE_CFG, "method": "qlora"}))
    g = g.to(device).eval()
    out, _ = gqla_hidden(g, ids)
    print(f"  qlora     4-bit base: cosine to fp32 {F.cosine_similarity(out.reshape(-1, out.size(-1)), ref.reshape(-1, ref.size(-1)), -1).mean().item():.4f}"
          f" | backbone weight memory {sum(b.numel() * b.element_size() for b in g.buffers())/2**20:.0f} MiB")
    del g; torch.cuda.empty_cache()


check_gqla_patch()

(1) patched vs original attention      max|diff| 5.36e-06
(2) cached decoding vs full forward     max|diff| 4.38e-06
(3) left-padded row vs unpadded row     max|diff| 4.65e-06  | NaN: False
(4) autocast vs fp32                   min cosine 1.00000
GQLA self-test passed


In [ ]:
# ============================================================
# 2. DOCUMENT MEMORY ("encoder" side of the GQLA summariser)
#
# The frozen Phase-1 GQLA reads every 512-token chunk once. Each chunk is compressed to m slot vectors =
# mean of the GQLA hidden states over m consecutive segments of the chunk (a causal state already summarises
# everything before it, so later segments carry more of the chunk). K chunks x m slots = the document memory,
# cached in fp16 on Drive. A small trainable bridge (bidirectional Transformer over the slots + projection into
# the token-embedding space) turns it into soft tokens that GQLA reads through ordinary self-attention:
#
#   [ prompt? | pad .. pad | mem_1 .. mem_N | <s> | y_1 .. y_L | </s> ]     (<= 512 positions)
#                                             loss only on y_1 .. </s>
# ============================================================
@torch.no_grad()
def extract_memory(docs, K, m, strategy, layer=None, tok_budget=32768, max_bs=128, log_every=50):
    bb = GQLA_BASE.to(device).eval()
    per_doc = [select_chunks(chunk_ids(d, GQLA_CTX_LEN, GQLA_CTX_LEN), K, strategy) for d in docs]
    flat = [c for cs in per_doc for c in cs]
    n, d = len(flat), bb.token_embedding.embedding_dim
    lens = np.fromiter((max(len(c), 1) for c in flat), dtype=np.int64, count=n)
    order = np.argsort(lens, kind="stable")
    out = torch.empty(n, m, d, dtype=torch.float16)
    print(f"  {n:,} chunks | {int(lens.sum()):,} tokens -> {m} slots per chunk")

    def run(sel):
        T = int(lens[sel].max())
        arr = np.full((len(sel), T), PAD_ID, dtype=np.int64)
        for j, i in enumerate(sel):
            c = flat[i]
            if len(c):
                arr[j, :len(c)] = c
        ids = torch.from_numpy(arr).to(device)
        ln = torch.from_numpy(lens[sel]).to(device)
        with amp_ctx():
            h, _ = gqla_hidden(bb, ids, upto=layer)                             # right padding is safe (causal)
        h = h.float()
        pos = torch.arange(T, device=device)[None, :]
        seg = (pos * m // ln[:, None]).clamp(max=m - 1)                         # segment id of every position
        oh = F.one_hot(seg, m).float() * (pos < ln[:, None]).unsqueeze(-1).float()   # [b,T,m]
        slots = torch.einsum("btm,btd->bmd", oh, h) / oh.sum(1).clamp_min(1).unsqueeze(-1)
        out[torch.from_numpy(sel)] = slots.half().cpu()

    def safe_run(sel):
        try:
            run(sel)
        except torch.cuda.OutOfMemoryError:
            if len(sel) == 1:
                raise
            torch.cuda.empty_cache(); safe_run(sel[: len(sel) // 2]); safe_run(sel[len(sel) // 2:])

    t0, s, nb = time.time(), 0, 0
    while s < n:
        e = s + 1
        while e < n and (e - s) < max_bs and (e + 1 - s) * lens[order[e]] <= tok_budget:
            e += 1
        safe_run(order[s:e]); s = e; nb += 1
        if nb % log_every == 0 or s >= n:
            print(f"    {s:,}/{n:,} chunks ({time.time() - t0:.0f}s)")
    feats, p = [], 0
    for cs in per_doc:
        feats.append(out[p:p + len(cs)].reshape(-1, d)); p += len(cs)        # [n_chunks*m, d]
    return feats


def get_memory(split, docs, cfg):
    name = f"mem_{split}_s{SPLIT_SEED}_K{cfg.mem_chunks}_m{cfg.mem_slots}_{cfg.chunk_strategy}_L{cfg.mem_layer}.pt"
    dp, lp = os.path.join(p6_dir, name), os.path.join("/content/feat_cache", name)
    os.makedirs("/content/feat_cache", exist_ok=True)
    if not os.path.exists(lp) and os.path.exists(dp):
        shutil.copy(dp, lp)
    if os.path.exists(lp):
        return torch.load(lp)
    print(f"extracting GQLA memory for '{split}' ...")
    feats = extract_memory(docs, cfg.mem_chunks, cfg.mem_slots, cfg.chunk_strategy,
                           None if cfg.mem_layer < 0 else cfg.mem_layer)
    torch.save(feats, lp); shutil.copy(lp, dp)
    return feats


class MemoryBridge(nn.Module):
    """slots [B,N,d] (left-padded) -> soft tokens in GQLA's input-embedding space.
    Adds chunk / slot position embeddings, mixes the slots bidirectionally (the causal GQLA features of one chunk
    never saw the other chunks), then projects and rescales to the token-embedding norm."""

    def __init__(self, d, n_chunks, n_slots, emb_std, layers=1, heads=8, dropout=0.1):
        super().__init__()
        self.n_slots = n_slots
        self.chunk_pos, self.slot_pos = nn.Embedding(n_chunks, d), nn.Embedding(n_slots, d)
        nn.init.normal_(self.chunk_pos.weight, std=0.02); nn.init.normal_(self.slot_pos.weight, std=0.02)
        self.ln_in = nn.LayerNorm(d)
        self.mix = (nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu",
                                                                     batch_first=True, norm_first=True),
                                          layers, enable_nested_tensor=False) if layers > 0 else None)
        self.ln_out, self.proj = nn.LayerNorm(d), nn.Linear(d, d)
        self.gain = nn.Parameter(torch.tensor(float(emb_std)))

    def forward(self, feats, mask, slot_idx):                 # slot_idx [B,N] = index of the slot inside its doc
        z = self.ln_in(feats) + self.chunk_pos(slot_idx // self.n_slots) + self.slot_pos(slot_idx % self.n_slots)
        if self.mix is not None:
            z = self.mix(z, src_key_padding_mask=~mask)
        return self.proj(self.ln_out(z)) * self.gain

In [ ]:
# ============================================================
# 3. GQLA SUMMARISER + ADAPTATION METHODS
# ============================================================
GQLA_LINEARS = ("W_DQ", "W_UQ", "W_DKV", "W_UK", "W_UV", "W_QR", "W_KR", "W_O", "feed_forward.0", "feed_forward.2")


def _is_target(name):
    return name.startswith("layers.") and any(name.endswith(t) for t in GQLA_LINEARS)


def apply_backbone_method(g, cfg):
    """Sets requires_grad / injects modules into the GQLA backbone g. Returns a short description."""
    meth = cfg.method
    freeze(g)
    d = g.token_embedding.embedding_dim
    if meth in ("full", "kd"):
        for p in g.parameters():
            p.requires_grad = True
        return "all GQLA weights"
    if meth == "partial":
        k = cfg.unfreeze_top_k
        for blk in g.layers[len(g.layers) - k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in g.norm_final.parameters():
            p.requires_grad = True
        return f"top {k} blocks + final norm"
    if meth == "frozen":
        return "backbone frozen (memory bridge only)"
    if meth == "lora":
        n = replace_linears(g, lambda l: LoRALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRA r={cfg.r} on {len(n)} linears"
    if meth == "lorma":
        n = replace_linears(g, lambda l: LoRMALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRMA+ r={cfg.r} on {len(n)} linears"
    if meth == "qlora":
        n = replace_linears(g, lambda l: LoRALinear(NF4Linear(l), cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"QLoRA (NF4 base) r={cfg.r} on {len(n)} linears"
    if meth in ("houlsby", "pfeiffer"):
        for blk in g.layers:
            if meth == "houlsby":
                blk.attn = AdapterAfter(blk.attn, d, cfg.adapter_dim, cfg.adapter_dropout)
            blk.feed_forward = AdapterAfter(blk.feed_forward, d, cfg.adapter_dim, cfg.adapter_dropout)
        return f"{meth} adapters (bottleneck {cfg.adapter_dim})"
    if meth == "prefix":
        for blk in g.layers:
            blk.attn.prefix = nn.Parameter(torch.randn(cfg.prefix_len, d) * 0.5)
        return f"prefix tuning, {cfg.prefix_len} prefix vectors per layer"
    if meth == "prompt":
        return f"prompt tuning, {cfg.prompt_len} soft tokens"
    raise ValueError(f"unknown method {meth!r}")


class GQLASummarizer(nn.Module):
    def __init__(self, gqla, cfg):
        super().__init__()
        self.g, self.cfg = gqla, cfg
        self.pretrained_param_ids = {id(p) for p in gqla.parameters()}        # before any injection
        self.method_desc = apply_backbone_method(gqla, cfg)
        d = gqla.token_embedding.embedding_dim
        emb_std = float(gqla.token_embedding.weight.detach().float().std())
        self.bridge = (MemoryBridge(d, cfg.mem_chunks, cfg.mem_slots, emb_std, cfg.bridge_layers)
                       if cfg.mem_mode == "memory" else None)
        self.prompt = None
        if cfg.method == "prompt":
            init = gqla.token_embedding.weight.detach()[torch.randint(1000, 20000, (cfg.prompt_len,))].clone()
            self.prompt = nn.Parameter(init)

    # ---- inputs: [prompt | memory (left-padded) | <s>] ----------------------------------------
    def _prefix_inputs(self, b):
        g = self.g
        emb = g.token_embedding
        if self.bridge is not None:
            mask = b["mem_mask"].to(device)
            mem = self.bridge(b["mem"].to(device).float(), mask, b["slot_idx"].to(device))
        else:
            mask = b["lead_mask"].to(device)
            mem = emb(b["lead"].to(device))
        B = mem.size(0)
        dt = emb.weight.dtype
        parts, kps = [mem.to(dt)], [mask]
        if self.prompt is not None:
            parts.insert(0, self.prompt.to(dt).unsqueeze(0).expand(B, -1, -1))
            kps.insert(0, mask.new_ones(B, self.prompt.size(0)))
        parts.append(emb(torch.full((B, 1), BOS_ID, dtype=torch.long, device=device)))
        kps.append(mask.new_ones(B, 1))
        return torch.cat(parts, 1), torch.cat(kps, 1)

    def _summary_states(self, b):
        pe, pk = self._prefix_inputs(b)
        y, ym, tgt = b["y"].to(device), b["y_mask"].to(device), b["tgt"].to(device)
        x = torch.cat([pe, self.g.token_embedding(y).to(pe.dtype)], 1)
        h, _ = gqla_hidden(self.g, embeds=x, key_pad=torch.cat([pk, ym], 1), ckpt=self.cfg.grad_ckpt)
        Tp, L = pe.size(1), y.size(1)
        hs = h[:, Tp - 1:Tp + L]                                  # position Tp-1 (<s>) predicts y_1 ... last predicts </s>
        sel = tgt != -100
        return hs[sel], tgt[sel]

    def logits_targets(self, b):
        h, t = self._summary_states(b)
        return self.g.lm_head(h), t

    def loss(self, b, teacher=None):
        logits, t = self.logits_targets(b)
        ce = F.cross_entropy(logits.float(), t, label_smoothing=self.cfg.label_smoothing)
        if teacher is None:
            return ce
        with torch.no_grad():
            tl, tt = teacher.logits_targets(b)
        assert tt.shape == t.shape
        T = self.cfg.kd_temp
        kl = F.kl_div(F.log_softmax(logits.float() / T, -1), F.log_softmax(tl.float() / T, -1),
                      log_target=True, reduction="batchmean") * (T * T)
        return (1 - self.cfg.kd_alpha) * ce + self.cfg.kd_alpha * kl

    @torch.no_grad()
    def generate(self, b, gen):
        g = self.g
        pe, pk = self._prefix_inputs(b)
        room = GQLA_CTX_LEN - pe.size(1) - 1
        max_new = min(gen["max_new"], room) if gen.get("respect_ctx", True) else gen["max_new"]

        def start():
            h, past = gqla_hidden(g, embeds=pe, key_pad=pk, use_cache=True)
            return {"past": past, "kp": pk}, g.lm_head(h[:, -1]).float()

        def step(st, tok):
            kp = torch.cat([st["kp"], st["kp"].new_ones(tok.size(0), 1)], 1)
            h, past = gqla_hidden(g, ids=tok[:, None], key_pad=kp, past=st["past"], use_cache=True)
            return {"past": past, "kp": kp}, g.lm_head(h[:, -1]).float()

        def reorder(st, idx):
            return {"past": reorder_cache(st["past"], idx), "kp": st["kp"].index_select(0, idx)}

        return beam_search(start, step, reorder, num_beams=gen["num_beams"], max_new=max_new,
                           min_new=min(gen["min_new"], max_new // 2), len_penalty=gen["len_penalty"],
                           no_repeat_ngram=gen["no_repeat_ngram"], rep_penalty=gen["rep_penalty"],
                           banned=(PAD_ID, BOS_ID))


def build_student_backbone(teacher_g, n_layers):
    """Smaller GQLA initialised from evenly spaced layers of the (fine-tuned) teacher (DistilBART-style
    'shrink and fine-tune'); embeddings, final norm and LM head are copied."""
    s = GQLAModel(vocab_size=len(tokenizer.itos), pad_token_id=PAD_ID, **{**GQLA_CFG, "n_layers": n_layers})
    keep = np.linspace(0, len(teacher_g.layers) - 1, n_layers).round().astype(int).tolist()
    s.token_embedding.load_state_dict(teacher_g.token_embedding.state_dict())
    s.norm_final.load_state_dict(teacher_g.norm_final.state_dict())
    s.lm_head.load_state_dict(teacher_g.lm_head.state_dict())
    for j, i in enumerate(keep):
        s.layers[j].load_state_dict(teacher_g.layers[i].state_dict())
    print(f"student: {n_layers} layers from teacher layers {keep} | "
          f"{sum(p.numel() for p in s.parameters())/1e6:.1f}M params (teacher "
          f"{sum(p.numel() for p in teacher_g.parameters())/1e6:.1f}M)")
    return s


INPUT_KEYS = ("mem_mode", "mem_chunks", "mem_slots", "chunk_strategy", "mem_layer", "bridge_layers", "max_sum_tok", "lead_tok")


def build_summarizer(cfg, backbone=None):
    g = copy.deepcopy(GQLA_BASE) if backbone is None else backbone
    return GQLASummarizer(g, cfg).to(device)


def make_student_summarizer(teacher, cfg):
    sm = build_summarizer(cfg, backbone=build_student_backbone(teacher.g, cfg.student_layers))
    if teacher.bridge is not None:
        sm.bridge.load_state_dict(teacher.bridge.state_dict())        # start from the teacher's trained bridge
    return sm

In [ ]:
# ============================================================
# 4. DATASETS / COLLATE
# ============================================================
SPLITS = {"train": (train_recs, tr_doc, tr_sum), "val": (val_recs, va_doc, va_sum), "test": (test_recs, te_doc, None)}
_DS = {}


class GQLASummDS(Dataset):
    def __init__(self, split, cfg):
        recs, docs, sums = SPLITS[split]
        self.ids = [r["id"] for r in recs]
        self.src = (get_memory(split, docs, cfg) if cfg.mem_mode == "memory"
                    else [np.asarray(d[:cfg.lead_tok], dtype=np.int64) for d in docs])
        self.sums = sums

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.src[i], (None if self.sums is None else self.sums[i])


def make_collate(cfg, with_target):
    max_y = cfg.max_sum_tok

    def collate(batch):
        B = len(batch)
        out = {"ids": [b[0] for b in batch]}
        if cfg.mem_mode == "memory":
            N = max(b[1].size(0) for b in batch); d = batch[0][1].size(1)
            mem = torch.zeros(B, N, d, dtype=torch.float16)
            mask = torch.zeros(B, N, dtype=torch.bool); slot = torch.zeros(B, N, dtype=torch.long)
            for i, (_, f, _) in enumerate(batch):                     # LEFT-pad: the memory ends right before <s>
                n = f.size(0)
                mem[i, N - n:] = f; mask[i, N - n:] = True; slot[i, N - n:] = torch.arange(n)
            out.update(mem=mem, mem_mask=mask, slot_idx=slot)
        else:
            N = max(max(len(b[1]), 1) for b in batch)
            lead = torch.full((B, N), PAD_ID, dtype=torch.long); mask = torch.zeros(B, N, dtype=torch.bool)
            for i, (_, t, _) in enumerate(batch):
                if len(t):
                    lead[i, N - len(t):] = torch.from_numpy(t); mask[i, N - len(t):] = True
                else:
                    mask[i, -1] = True
            out.update(lead=lead, lead_mask=mask)
        if with_target:
            ys = [np.asarray(b[2][:max_y], dtype=np.int64) for b in batch]
            L = max(max(len(y) for y in ys), 1)
            y = torch.full((B, L), PAD_ID, dtype=torch.long); ym = torch.zeros(B, L, dtype=torch.bool)
            tgt = torch.full((B, L + 1), -100, dtype=torch.long)
            for i, s in enumerate(ys):
                n = len(s)
                y[i, :n] = torch.from_numpy(s); ym[i, :n] = True
                tgt[i, :n] = torch.from_numpy(s); tgt[i, n] = EOS_ID
            out.update(y=y, y_mask=ym, tgt=tgt)
        return out
    return collate


def make_loader(split, cfg, shuffle, gen=False, limit=None):
    key = (split, cfg.mem_mode, cfg.mem_chunks, cfg.mem_slots, cfg.chunk_strategy, cfg.mem_layer, cfg.lead_tok)
    if key not in _DS:
        _DS[key] = GQLASummDS(split, cfg)
    ds = _DS[key]
    if limit:
        ds = torch.utils.data.Subset(ds, list(range(min(limit, len(ds)))))
    elif split == "train" and cfg.max_train_docs:
        ds = torch.utils.data.Subset(ds, list(range(min(cfg.max_train_docs, len(ds)))))
    has_y = split != "test" and not gen
    return DataLoader(ds, batch_size=cfg.gen_bs if gen else cfg.batch_size, shuffle=shuffle,
                      collate_fn=make_collate(cfg, has_y), num_workers=0, drop_last=False)

In [ ]:
# ============================================================
# 5. CONFIGURATION
# ============================================================
BASE_CFG = dict(
    method="lora",
    # document memory: K chunks x m slots soft tokens (16 x 8 = 128 of the 512 positions)
    mem_mode="memory",            # "memory" (bridge over cached GQLA states) | "lead" (first tokens of the doc as text)
    mem_chunks=16, mem_slots=8, chunk_strategy="uniform", mem_layer=-1, bridge_layers=1,
    lead_tok=None,                # lead mode: filled in by make_cfg to use the whole remaining window
    max_sum_tok=None,             # filled in by make_cfg: min(p95 of reference lengths, room left in 512)
    # adaptation hyper-parameters
    r=16, alpha=32, lora_dropout=0.05, adapter_dim=64, adapter_dropout=0.1,
    prefix_len=16, prompt_len=20, unfreeze_top_k=3,
    # optimisation
    lr=5e-4, lr_backbone=5e-5, wd=0.01, epochs=6, patience=2, batch_size=8, accum=2, warmup=0.05, clip=1.0,
    label_smoothing=0.1, grad_ckpt=True, seed=42, log_every=100, max_train_docs=None,
    # model selection during training (generation on a slice of val is the honest signal, not loss)
    val_loss_batches=20, n_eval_gen=64, select_metric="rougeL", gen_bs=8, n_eval_final=None,
    # knowledge distillation
    teacher_ckpt=None, student_layers=5, kd_alpha=0.5, kd_temp=2.0,
)

METHOD_DEFAULTS = {                                    # sensible starting points per method
    "full":     dict(lr=5e-4, lr_backbone=3e-5),
    "partial":  dict(lr=5e-4, lr_backbone=5e-5),
    "frozen":   dict(lr=1e-3),
    "lora":     dict(lr=5e-4),
    "lorma":    dict(lr=5e-4),
    "qlora":    dict(lr=5e-4),
    "houlsby":  dict(lr=5e-4),
    "pfeiffer": dict(lr=5e-4),
    "prefix":   dict(lr=2e-3),
    "prompt":   dict(lr=3e-3),
    "kd":       dict(lr=5e-4, lr_backbone=5e-5),
}


def make_cfg(**over):
    meth = over.get("method", BASE_CFG["method"])
    c = SimpleNamespace(**{**BASE_CFG, **METHOD_DEFAULTS.get(meth, {}), **over})
    P = c.prompt_len if c.method == "prompt" else 0
    if c.mem_mode == "memory":
        n_mem = c.mem_chunks * c.mem_slots
        room = GQLA_CTX_LEN - P - n_mem - 1 - 1                # -1 for <s>, -1 for </s>
        c.max_sum_tok = c.max_sum_tok or min(SUM_TOK_P95, room)
        assert c.max_sum_tok <= room, f"summary budget {c.max_sum_tok} > room {room}"
    else:
        c.max_sum_tok = c.max_sum_tok or min(SUM_TOK_P90, GQLA_CTX_LEN // 2)
        c.lead_tok = c.lead_tok or GQLA_CTX_LEN - P - 1 - c.max_sum_tok - 1
    return c


GEN = dict(num_beams=4, max_new=SUM_TOK_P90, min_new=SUM_TOK_P10, len_penalty=1.0,
           no_repeat_ngram=3, rep_penalty=1.0, max_words=None, respect_ctx=True)

_c = make_cfg()
print(f"memory layout: {_c.mem_chunks} chunks x {_c.mem_slots} slots = {_c.mem_chunks * _c.mem_slots} soft tokens "
      f"| summary budget {_c.max_sum_tok} tokens (reference p95 = {SUM_TOK_P95})")
_trunc = (sl > _c.max_sum_tok).mean()
if _trunc > 0.05:
    print(f"NOTE: {_trunc:.0%} of training summaries are longer than the {_c.max_sum_tok}-token budget and are "
          f"truncated; GQLA's 512-token window is the limit here. Fewer memory slots (mem_chunks / mem_slots) "
          f"buy more summary room.")

check_peft_identity()

memory layout: 16 chunks x 8 slots = 128 soft tokens | summary budget 382 tokens (reference p95 = 2398)
NOTE: 86% of training summaries are longer than the 382-token budget and are truncated; GQLA's 512-token window is the limit here. Fewer memory slots (mem_chunks / mem_slots) buy more summary room.
  lora      identity at init: max|diff| 0.00e+00
  lorma     identity at init: max|diff| 4.29e-06
  houlsby   identity at init: max|diff| 0.00e+00
  pfeiffer  identity at init: max|diff| 0.00e+00
  qlora     4-bit base: cosine to fp32 0.9918 | backbone weight memory 36 MiB


In [ ]:
# ============================================================
# 6. EXPERIMENT RUNNER
# ============================================================
RESULTS = {}
_res_path = os.path.join(p6_dir, "results.json")
if os.path.exists(_res_path):
    RESULTS = json.load(open(_res_path))


def _teacher_from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    tcfg = SimpleNamespace(**ck["cfg"])
    t = build_summarizer(tcfg)
    load_trainable(t, path)
    return t, tcfg


def evaluate_and_submit(sm, cfg, name, submit=True):
    t0 = time.time()
    val_pred = generate_split(sm, "val", cfg, n_docs=getattr(cfg, "n_eval_final", None))
    m = score_split(val_pred, val_recs, bert=True)
    print(f"[{name}] VAL ({len(val_pred)} docs): {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
    RESULTS[name] = dict(val=m, method=cfg.method, desc=sm.method_desc,
                         trainable=n_params(sm), total=n_params(sm, trainable=False),
                         cfg={k: v for k, v in vars(cfg).items() if isinstance(v, (int, float, str, bool, type(None)))})
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    json.dump(val_pred, open(os.path.join(p6_dir, f"valpred_{name}.json"), "w"), ensure_ascii=False)
    if submit:
        test_pred = generate_split(sm, "test", cfg)
        write_submission(test_pred, name)
    return m


def run_experiment(name, submit=True, **over):
    cfg = make_cfg(**over)
    set_seed(cfg.seed)
    teacher = None
    if cfg.method == "kd":
        assert cfg.teacher_ckpt and os.path.exists(cfg.teacher_ckpt), "kd needs teacher_ckpt (e.g. the full fine-tune)"
        teacher, tcfg = _teacher_from_ckpt(cfg.teacher_ckpt)
        for k in INPUT_KEYS:
            setattr(cfg, k, getattr(tcfg, k))                       # student reads exactly the teacher's inputs
        sm = make_student_summarizer(teacher, cfg)
    else:
        sm = build_summarizer(cfg)
    print(f"\n=== {name} | {sm.method_desc} | trainable {n_params(sm)/1e6:.2f}M / {n_params(sm, False)/1e6:.1f}M ===")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_summarizer(sm, cfg, name, teacher=teacher)
    train_min = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0
    del teacher; gc.collect(); torch.cuda.empty_cache()
    m = evaluate_and_submit(sm, cfg, name, submit)
    RESULTS[name].update(train_min=train_min, peak_gb=peak)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    del sm; gc.collect(); torch.cuda.empty_cache()
    return m


def resubmit(name, **gen_over):
    """Re-generate val + test from a saved checkpoint with different decoding (no retraining)."""
    path = os.path.join(p6_dir, f"ckpt_{name}.pt")
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = SimpleNamespace(**ck["cfg"])
    if cfg.method == "kd":
        teacher, _ = _teacher_from_ckpt(cfg.teacher_ckpt)
        sm = make_student_summarizer(teacher, cfg); del teacher
    else:
        sm = build_summarizer(cfg)
    load_trainable(sm, path)
    saved = dict(GEN); GEN.update(gen_over)
    tag = name + "_" + "_".join(f"{k}{v}" for k, v in gen_over.items())
    try:
        return evaluate_and_submit(sm, cfg, tag)
    finally:
        GEN.clear(); GEN.update(saved); del sm; gc.collect(); torch.cuda.empty_cache()


def show_results():
    print(f"\n{'experiment':26s} {'method':9s} {'BERTScore':>9s} {'R1':>7s} {'R2':>7s} {'RL':>7s} {'words':>6s} "
          f"{'train M':>8s} {'total M':>8s} {'min':>6s} {'GB':>5s}")
    for n, r in RESULTS.items():
        v = r["val"]; bs = v.get("bertscore")
        print(f"{n:26s} {r['method']:9s} {('%.4f' % bs) if bs is not None else 'n/a':>9s} {v['rouge1']:7.4f} "
              f"{v['rouge2']:7.4f} {v['rougeL']:7.4f} {v['avg_words']:6.0f} {r['trainable']/1e6:8.2f} "
              f"{r['total']/1e6:8.1f} {r.get('train_min', 0):6.0f} {r.get('peak_gb', 0):5.1f}")

In [ ]:
# ============================================================
# 7. RUN
# ============================================================
# (0) Smoke test FIRST (a few minutes): the whole pipeline end-to-end on 64 documents, no submission.
#     It extracts + caches the GQLA memory for train/val/test on first use (one-off, ~10-20 min on a T4).
run_experiment("G0_smoke", method="lora", max_train_docs=64, epochs=1, n_eval_gen=8, n_eval_final=16, submit=False)


# (2) Optional extras / ablations
# run_experiment("G7_qlora",    method="qlora")                            # 4-bit base: memory vs quality
# run_experiment("G8_pfeiffer", method="pfeiffer")
# run_experiment("G9_prompt",   method="prompt")
# run_experiment("G10_frozen",  method="frozen")                           # bridge only: how much does GQLA adapt?
# run_experiment("G11_lead",    method="lora", mem_mode="lead")            # design A: plain text prefix, no memory
# run_experiment("G12_head",    method="lora", chunk_strategy="head_tail") # which parts of the judgment matter
# run_experiment("G13_m4",      method="lora", mem_slots=4)                # shorter memory, longer summaries

# (3) Decoding sweeps reuse a trained checkpoint (no retraining), e.g. length:
# resubmit("G3_lorma", len_penalty=1.2, min_new=200)
# resubmit("G3_lorma", num_beams=1)

show_results()


=== G0_smoke | LoRA r=16 on 100 linears | trainable 6.84M / 110.7M ===
extracting GQLA memory for 'train' ...
  57,262 chunks | 27,713,751 tokens -> 8 slots per chunk
    5,503/57,262 chunks (25s)
    8,763/57,262 chunks (63s)
    11,963/57,262 chunks (103s)
    15,163/57,262 chunks (147s)
    18,363/57,262 chunks (195s)
    21,563/57,262 chunks (243s)
    24,763/57,262 chunks (291s)
    27,963/57,262 chunks (339s)
    31,163/57,262 chunks (387s)
    34,363/57,262 chunks (436s)
    37,563/57,262 chunks (483s)
    40,763/57,262 chunks (531s)
    43,963/57,262 chunks (579s)
    47,163/57,262 chunks (627s)
    50,363/57,262 chunks (675s)
    53,563/57,262 chunks (723s)
    56,763/57,262 chunks (771s)
    57,262/57,262 chunks (779s)
extracting GQLA memory for 'val' ...
  3,024 chunks | 1,464,979 tokens -> 8 slots per chunk
    3,024/3,024 chunks (44s)
[G0_smoke] 8 batches/epoch x 1 epochs | accum 2 | trainable new 6.84M + backbone 0.00M


/tmp/ipykernel_4337/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run += float(loss)


[G0_smoke] ep 01 train 6.8534 | val loss 6.4645 | BERTScore n/a | R1 0.1005 R2 0.0270 RL 0.0923 | words 133 (ref 501) (44s)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[G0_smoke] VAL (16 docs): BERTScore 0.7316 | R1 0.0984 R2 0.0260 RL 0.0886 | words 115 (ref 521) (92s)

experiment                 method    BERTScore      R1      R2      RL  words  train M  total M    min    GB
G0_smoke                   lora         0.7316  0.0984  0.0260  0.0886    115     6.84    110.7     15   3.7


In [ ]:
# (1) Six substantially different adaptation methods (LoRMA compulsory, KD mandatory). Same memory input,
#     same decoding, so differences come from the adaptation method.
run_experiment("G1_full",    method="full", batch_size=4, accum=4)        # also the KD teacher
run_experiment("G2_lora",    method="lora")
run_experiment("G3_lorma",   method="lorma")                             # compulsory
run_experiment("G4_houlsby", method="houlsby")
run_experiment("G5_prefix",  method="prefix")
run_experiment("G6_kd",      method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_G1_full.pt"), student_layers=5)


=== G1_full | all GQLA weights | trainable 107.32M / 107.3M ===
[G1_full] 1670 batches/epoch x 6 epochs | accum 4 | trainable new 3.43M + backbone 103.89M
[G1_full] ep 1 batch 100/1670 loss 6.8146 (35s)
[G1_full] ep 1 batch 200/1670 loss 6.3440 (69s)
[G1_full] ep 1 batch 300/1670 loss 5.9850 (103s)
[G1_full] ep 1 batch 400/1670 loss 5.7302 (137s)
[G1_full] ep 1 batch 500/1670 loss 5.5510 (171s)
[G1_full] ep 1 batch 600/1670 loss 5.4115 (205s)
[G1_full] ep 1 batch 700/1670 loss 5.3002 (239s)
[G1_full] ep 1 batch 800/1670 loss 5.2146 (273s)
[G1_full] ep 1 batch 900/1670 loss 5.1412 (307s)
[G1_full] ep 1 batch 1000/1670 loss 5.0798 (342s)
[G1_full] ep 1 batch 1100/1670 loss 5.0276 (376s)
[G1_full] ep 1 batch 1200/1670 loss 4.9827 (410s)
[G1_full] ep 1 batch 1300/1670 loss 4.9410 (444s)
[G1_full] ep 1 batch 1400/1670 loss 4.9047 (478s)
[G1_full] ep 1 batch 1500/1670 loss 4.8729 (512s)
[G1_full] ep 1 batch 1600/1670 loss 4.8439 (546s)
[G1_full] ep 01 train 4.8251 | val loss 4.2603 | BERTSc

{'rouge1': 0.3984355223750186,
 'rouge2': 0.1622866516362638,
 'rougeL': 0.2080203774204857,
 'bertscore': 0.8348249793052673,
 'avg_words': 287.9344729344729,
 'ref_words': 824.957264957265}